<a href="https://colab.research.google.com/github/sreeja99sunkara/FitAgent/blob/main/fitness_coach_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install smolagents -U

In [ ]:
!pip install -U smolagents ddgs

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from smolagents import CodeAgent, tool

@tool
def bmi_calculator(weight: float, height: float) -> str:
    """
    Calculates BMI.

    Args:
        weight: Weight in kg.
        height: Height in cm.
    """
    bmi = weight / ((height/100)**2)
    return f"BMI = {bmi:.2f}"

In [ ]:
from smolagents import tool

@tool
def calorie_calculator(
    weight: float,
    height: float,
    age: int,
    gender: str,
    goal: str
) -> str:
    """
    Calculates daily calorie requirements.

    Args:
        weight: Weight in kilograms.
        height: Height in centimeters.
        age: Age in years.
        gender: male or female.
        goal: weight_loss, maintenance, or muscle_gain.
    """

    gender = gender.lower()
    goal = goal.lower()

    # Mifflin-St Jeor Equation
    if gender == "male":
        bmr = 10 * weight + 6.25 * height - 5 * age + 5
    else:
        bmr = 10 * weight + 6.25 * height - 5 * age - 161

    # Assuming moderate activity
    maintenance = bmr * 1.55

    if goal == "weight_loss":
        target = maintenance - 500
    elif goal == "muscle_gain":
        target = maintenance + 300
    else:
        target = maintenance

    return (
        f"BMR: {bmr:.0f} kcal/day\n"
        f"Maintenance Calories: {maintenance:.0f} kcal/day\n"
        f"Recommended Calories for {goal}: {target:.0f} kcal/day"
    )

In [ ]:
from smolagents import tool

@tool
def workout_generator(goal: str) -> str:
    """
    Generates a workout plan based on fitness goal.

    Args:
        goal: weight_loss, muscle_gain, or maintenance.
    """

    goal = goal.lower()

    workouts = {
        "weight_loss": """
Monday: 30 min Cardio + Full Body Workout
Tuesday: Walking + Core Exercises
Wednesday: HIIT (20 mins)
Thursday: Strength Training
Friday: Cardio + Abs
Saturday: Outdoor Activity
Sunday: Rest
""",

        "muscle_gain": """
Monday: Chest + Triceps
Tuesday: Back + Biceps
Wednesday: Legs
Thursday: Shoulders
Friday: Upper Body
Saturday: Lower Body
Sunday: Rest
""",

        "maintenance": """
Monday: Full Body Workout
Tuesday: Cardio
Wednesday: Yoga/Stretching
Thursday: Strength Training
Friday: Cardio
Saturday: Sports Activity
Sunday: Rest
"""
    }

    return workouts.get(goal, "Please choose: weight_loss, muscle_gain, or maintenance.")


In [ ]:
from smolagents import tool

@tool
def diet_suggestion(goal: str, diet_type: str = "vegetarian") -> str:
    """
    Suggests a diet plan.

    Args:
        goal: weight_loss, muscle_gain, or maintenance.
        diet_type: vegetarian or non_vegetarian.
    """

    goal = goal.lower()
    diet_type = diet_type.lower()

    diets = {
        ("weight_loss", "vegetarian"):
            """
Breakfast: Oats + Fruits
Lunch: Brown Rice + Dal + Salad
Snack: Green Tea + Nuts
Dinner: Vegetable Soup + Paneer
""",

        ("weight_loss", "non_vegetarian"):
            """
Breakfast: Eggs + Fruits
Lunch: Grilled Chicken + Salad
Snack: Nuts
Dinner: Fish + Vegetables
""",

        ("muscle_gain", "vegetarian"):
            """
Breakfast: Milk + Peanut Butter Toast
Lunch: Rice + Paneer + Dal
Snack: Protein Shake
Dinner: Chapati + Soy Chunks
""",

        ("muscle_gain", "non_vegetarian"):
            """
Breakfast: Eggs + Milk
Lunch: Chicken Breast + Rice
Snack: Protein Shake
Dinner: Fish + Sweet Potato
""",

        ("maintenance", "vegetarian"):
            """
Balanced meals with vegetables,
whole grains, fruits and protein.
""",

        ("maintenance", "non_vegetarian"):
            """
Balanced meals with lean meat,
vegetables and whole grains.
"""
    }

    return diets.get(
        (goal, diet_type),
        "Please choose valid goal and diet type."
    )

In [ ]:
from smolagents import InferenceClientModel, DuckDuckGoSearchTool

search_tool = DuckDuckGoSearchTool()

agent = CodeAgent(
    tools=[
        bmi_calculator,
        calorie_calculator,
        workout_generator,
        diet_suggestion,
        search_tool
    ],
    model=InferenceClientModel()
)

agent.run("""I am 20 years old, female, 52kg, 172cm.
    My goal is muscle gain.
    Give me:
    1. BMI
    2. Daily calories
    3. Workout plan
    4. Diet plan
    5. Top 5 high-protein non-vegetarian foods
    """)